### 01 - Instalação (Biblioteca)

In [ ]:
%pip install numpy

### 02 - Importação (Recursos)

In [ ]:
import numpy as np
import csv

print(f"Numpy Version: {np.__version__}\n")
print(f"CSV Version: {csv.__version__}")

### 03 - Leitura do CSV

In [ ]:
def load_csv_data(file_path):
    X = []
    y = []

    with open(file_path, "r", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            passenger_survived = row["survived"]

            if (passenger_survived == ""):
                continue

            y.append(int(passenger_survived))

            passenger_class = float(row["pclass"]) if row["pclass"] != "" else 0

            passenger_sex = 1 if row["sex"] == "female" else 0

            passenger_age = float(row["age"]) if row["age"] != "" else -1

            passenger_sibsp = float(row["sibsp"]) if (row["sibsp"]) != "" else 0

            passenger_parch = float(row["parch"]) if row["parch"] != "" else 0

            passenger_fare = float(row["fare"]) if row["fare"] != "" else 0

            X.append([
                passenger_class,
                passenger_sex,
                passenger_age,
                passenger_sibsp,
                passenger_parch,
                passenger_fare
            ])

    return np.array(X), np.array(y).reshape(-1, 1)

### 04 - Tratamento de Dados

In [ ]:
def fill_missing_age(X):
    ages = X[:, 2]

    mean_age = np.mean(ages[ages != -1])

    X[:, 2] = np.where(ages == -1, mean_age, ages)

    return X

def normalize(X):
    mean = np.mean(X, axis=0)

    std = np.std(X, axis=0)

    return (X - mean) / (std + 1e-8)

def add_bias(X):
    ones = np.ones((X.shape[0], 1))

    return np.hstack((ones, X))

### 05 - Funções do Modelo

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_cost(X, y, beta):
    m = len(y)

    h = sigmoid(X @ beta)

    epsilson = 1e-8

    cost = -(1/m) * np.sum(
        y * np.log(h + epsilson) + (1 - y) * np.log(1 - h + epsilson)
    )

    return cost

def gradient_descent(X, y, beta, lr, epochs):
    m = len(y)

    for i in range(0, epochs):
        h = sigmoid(X @ beta)

        gradient = (1/m) * (X.T @ (h - y))

        beta = beta - lr * gradient

        if i % 100 == 0:
            print(f"Custo da Época ({i}): {compute_cost(X, y, beta):.4f}")

    return beta

### 06 - Funções de Processos

In [ ]:
def train(X, y):
    X = fill_missing_age(X)
    X = normalize(X)
    X = add_bias(X)

    beta = np.zeros((X.shape[1], 1))

    beta = gradient_descent(X, y, beta, lr=0.01, epochs=2000)

    return beta, X

def predict(X, beta):
    probability = sigmoid(X @ beta)

    return (probability >= 0.5).astype(int)

def confusion_matrix(y_true, y_pred):
    y_t = y_true.flatten()

    y_p = y_pred.flatten()

    VP = np.sum((y_t == 1) & (y_p == 1))
    VN = np.sum((y_t == 0) & (y_p == 0))
    FP = np.sum((y_t == 0) & (y_p == 1))
    FN = np.sum((y_t == 1) & (y_p == 0))

    print(f"\nVerdadeiros Positivos: {VP}")
    print(f"Verdadeiros Negativos: {VN}")
    print(f"Falsos Positivos: {FP}")
    print(f"Falsos Negativos: {FN}")

    return VP, VN, FP, FN

def calculate_metrics(VP, VN, FP, FN):
    accuracy = (VP + VN) / (VP + VN + FP + FN)

    precision = VP / (VP + FP)

    recall = VP / (VP + FN)

    specificity = VN / (VN + FP)

    f1_score = 2 * (precision * recall) / (precision + recall)

    print(f"\nAcurácia: {(accuracy * 100):.2f}%")
    print(f"Precisão: {(precision * 100):.2f}%")
    print(f"Revocação: {(recall * 100):.2f}%")
    print(f"Especificidade: {(specificity * 100):.2f}%")
    print(f"F1 Score: {(f1_score * 100):.2f}%")

    return accuracy, precision, recall, specificity, f1_score

### 07 - Função de Testes

In [ ]:
def train_test_split(X, y, test_size=0.2):
    np.random.seed(42)

    indexes = np.arange(len(X))

    np.random.shuffle(indexes)

    split = int(len(X) * (1 - test_size))

    train_indexes = indexes[:split]

    test_indexes = indexes[split:]

    return X[train_indexes], X[test_indexes], y[train_indexes], y[test_indexes]

### 08 - Execução

In [ ]:
X, y = load_csv_data("./titanic.csv")

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, X_train_processed = train(X_train, y_train)

X_test = fill_missing_age(X_test)
X_test = normalize(X_test)
X_test = add_bias(X_test)

y_pred = predict(X_test, beta)

VP, VN, FP, FN = confusion_matrix(y_test, y_pred)

accuracy, precision, recall, specificity, f1_score = calculate_metrics(VP, VN, FP, FN)